In [4]:
# ============================================================================
# STAGE 3 - CELL 1: SETUP & DATA LOADING
# ============================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("\n" + "=" * 80)
print("STAGE 3: BASELINE & FIRST MODEL TRAINING")
print("=" * 80)

print(f"\n📂 CELL 1: SETUP & DATA LOADING")
print("=" * 80)

# Load the cleaned datasets from corrected path
base_path = 'kcet_ml_project/data/stage2_v2_corrected/'

print(f"\n⏳ Loading datasets from {base_path} ...")
train_data = pd.read_csv(base_path + 'train_stage2_final.csv')
val_data = pd.read_csv(base_path + 'val_stage2_final.csv')
test_data = pd.read_csv(base_path + 'test_stage2_final.csv')

print(f"✅ Datasets loaded!")

# Separate features (X) and target (y)
print(f"\n⏳ Separating features and target...")
X_train = train_data.drop('Cutoff_Rank', axis=1)
y_train = train_data['Cutoff_Rank']

X_val = val_data.drop('Cutoff_Rank', axis=1)
y_val = val_data['Cutoff_Rank']

X_test = test_data.drop('Cutoff_Rank', axis=1)
y_test = test_data['Cutoff_Rank']

print(f"✅ Features and targets separated!")

# Data verification
print(f"\n" + "=" * 80)
print(f"📊 DATA VERIFICATION")
print(f"=" * 80)

print(f"\n🎯 Dataset Shapes:")
print(f"   Train: X={X_train.shape}, y={y_train.shape}")
print(f"   Val:   X={X_val.shape}, y={y_val.shape}")
print(f"   Test:  X={X_test.shape}, y={y_test.shape}")

print(f"\n📈 Feature Count:")
print(f"   Total features: {X_train.shape[1]}")
print(f"   All numeric: {X_train.select_dtypes(include=[np.number]).shape[1] == X_train.shape[1]}")

print(f"\n🎯 Target Statistics (Cutoff_Rank):")
print(f"   Train: mean={y_train.mean():,.0f}, std={y_train.std():,.0f}, min={y_train.min():,.0f}, max={y_train.max():,.0f}")
print(f"   Val:   mean={y_val.mean():,.0f}, std={y_val.std():,.0f}, min={y_val.min():,.0f}, max={y_val.max():,.0f}")
print(f"   Test:  mean={y_test.mean():,.0f}, std={y_test.std():,.0f}, min={y_test.min():,.0f}, max={y_test.max():,.0f}")

print(f"\n❌ Missing values:")
print(f"   Train X: {X_train.isnull().sum().sum()}, Train y: {y_train.isnull().sum()}")
print(f"   Val X:   {X_val.isnull().sum().sum()}, Val y:   {y_val.isnull().sum()}")
print(f"   Test X:  {X_test.isnull().sum().sum()}, Test y:  {y_test.isnull().sum()}")

print(f"\n✅ CELL 1 COMPLETE!")
print("=" * 80)



STAGE 3: BASELINE & FIRST MODEL TRAINING

📂 CELL 1: SETUP & DATA LOADING

⏳ Loading datasets from kcet_ml_project/data/stage2_v2_corrected/ ...
✅ Datasets loaded!

⏳ Separating features and target...
✅ Features and targets separated!

📊 DATA VERIFICATION

🎯 Dataset Shapes:
   Train: X=(137755, 32), y=(137755,)
   Val:   X=(60681, 32), y=(60681,)
   Test:  X=(71626, 32), y=(71626,)

📈 Feature Count:
   Total features: 32
   All numeric: True

🎯 Target Statistics (Cutoff_Rank):
   Train: mean=69,321, std=45,187, min=90, max=183,210
   Val:   mean=81,605, std=51,665, min=169, max=203,368
   Test:  mean=109,606, std=67,796, min=193, max=274,884

❌ Missing values:
   Train X: 0, Train y: 0
   Val X:   0, Val y:   0
   Test X:  0, Test y:  0

✅ CELL 1 COMPLETE!


In [5]:
# ============================================================================
# STAGE 3 - CELL 1.5: DIAGNOSTIC - FIND NON-NUMERIC COLUMNS
# ============================================================================

print("\n" + "=" * 80)
print("CELL 1.5: DIAGNOSTIC - FIND NON-NUMERIC COLUMNS")
print("=" * 80)

# Find all non-numeric columns in X_train
non_numeric_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\n❌ NON-NUMERIC COLUMNS FOUND: {len(non_numeric_cols)}")

if len(non_numeric_cols) > 0:
    for i, col in enumerate(non_numeric_cols, 1):
        dtype = X_train[col].dtype
        unique = X_train[col].nunique()
        print(f"   {i}. {col:<40} dtype={dtype}, unique={unique}")
else:
    print(f"   None - All columns are numeric!")

# Show all column names for reference
print(f"\n📋 ALL FEATURES IN X_train ({X_train.shape[1]} total):")
for i, col in enumerate(X_train.columns, 1):
    dtype = X_train[col].dtype
    print(f"   {i:2d}. {col:<40} {dtype}")

print("=" * 80)



CELL 1.5: DIAGNOSTIC - FIND NON-NUMERIC COLUMNS

❌ NON-NUMERIC COLUMNS FOUND: 0
   None - All columns are numeric!

📋 ALL FEATURES IN X_train (32 total):
    1. Year                                     int64
    2. Round                                    int64
    3. Exam_Type                                int64
    4. Years_Since_2020                         int64
    5. Is_Recent                                int64
    6. Year_Squared                             int64
    7. Historical_Mean_Primary                  float64
    8. Historical_Mean_Percentile               float64
    9. College_Tier_Numeric                     int64
   10. Category_Score                           float64
   11. Historical_Std_Raw                       float64
   12. Volatility_Category                      int64
   13. Historical_Count_Raw                     int64
   14. Program_Maturity                         int64
   15. Is_Established                           int64
   16. Branch_Popularity   

In [6]:
# ============================================================================
# STAGE 3 - CELL 2: CLEAN FEATURES & LOG TRANSFORM TARGET
# ============================================================================

print("\n" + "=" * 80)
print("CELL 2: CLEAN FEATURES & LOG TRANSFORM TARGET")
print("=" * 80)

# 'year_cohort' column is metadata and was not present in train features per previous cell,
# but if it does exist, drop it here to avoid issues
if 'year_cohort' in X_train.columns:
    print(f"\n🗑️  DROPPING NON-NUMERIC COLUMNS:")
    print(f"   Dropping: year_cohort (metadata)")
    X_train = X_train.drop(columns=['year_cohort'], errors='ignore')
    X_val = X_val.drop(columns=['year_cohort'], errors='ignore')
    X_test = X_test.drop(columns=['year_cohort'], errors='ignore')
    print(f"   ✅ Dropped!")

# Confirm all features are numeric
all_numeric = X_train.select_dtypes(include=[np.number]).shape[1] == X_train.shape[1]
print(f"\n✅ VERIFICATION: All features numeric? {all_numeric} (Features: {X_train.shape[1]})")

# Log transform targets to handle skewed distributions
print(f"\n📊 LOG TRANSFORM TARGET:")
print(f"   Using: log1p(Cutoff_Rank) for modeling")

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
y_test_log = np.log1p(y_test)

print(f"   ✅ Log transformation applied!")

# Show basic target statistics before/after transformation
print(f"\n📈 TARGET STATISTICS (Original Scale):")
print(f"   Train: mean={y_train.mean():,.0f}, std={y_train.std():,.0f}")
print(f"   Val:   mean={y_val.mean():,.0f}, std={y_val.std():,.0f}")
print(f"   Test:  mean={y_test.mean():,.0f}, std={y_test.std():,.0f}")

print(f"\n📈 TARGET STATISTICS (Log1p Scale):")
print(f"   Train: mean={y_train_log.mean():.3f}, std={y_train_log.std():.3f}")
print(f"   Val:   mean={y_val_log.mean():.3f}, std={y_val_log.std():.3f}")
print(f"   Test:  mean={y_test_log.mean():.3f}, std={y_test_log.std():.3f}")

print(f"\n✅ CELL 2 COMPLETE!")
print("=" * 80)



CELL 2: CLEAN FEATURES & LOG TRANSFORM TARGET

✅ VERIFICATION: All features numeric? True (Features: 32)

📊 LOG TRANSFORM TARGET:
   Using: log1p(Cutoff_Rank) for modeling
   ✅ Log transformation applied!

📈 TARGET STATISTICS (Original Scale):
   Train: mean=69,321, std=45,187
   Val:   mean=81,605, std=51,665
   Test:  mean=109,606, std=67,796

📈 TARGET STATISTICS (Log1p Scale):
   Train: mean=10.850, std=0.901
   Val:   mean=11.028, std=0.879
   Test:  mean=11.335, std=0.861

✅ CELL 2 COMPLETE!


In [15]:
# ============================================================================
# STAGE 3 - CELL 3: BASELINE MODELS (GLOBAL + LAG-1 BASELINE)
# ============================================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("\n" + "=" * 80)
print("CELL 3: BASELINE MODELS (STRONGER & CORRECT BENCHMARKS)")
print("=" * 80)

def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# ============================================================================
# BASELINE 1: GLOBAL MEAN
# ============================================================================

print("\n📊 BASELINE 1: Global Mean (Simple Average)")
print("=" * 60)

global_mean = y_train.mean()

baseline1_val_pred = np.full(len(y_val), global_mean)
baseline1_test_pred = np.full(len(y_test), global_mean)

baseline1_val_mae = mean_absolute_error(y_val, baseline1_val_pred)
baseline1_test_mae = mean_absolute_error(y_test, baseline1_test_pred)

baseline1_val_rmse = calculate_rmse(y_val, baseline1_val_pred)
baseline1_test_rmse = calculate_rmse(y_test, baseline1_test_pred)

print(f"   Global mean: {global_mean:,.0f}")
print(f"   Validation MAE:  {baseline1_val_mae:,.0f}")
print(f"   Validation RMSE: {baseline1_val_rmse:,.0f}")
print(f"   Test MAE:        {baseline1_test_mae:,.0f}")
print(f"   Test RMSE:       {baseline1_test_rmse:,.0f}")

# ============================================================================
# BASELINE 2: GLOBAL MEDIAN
# ============================================================================

print("\n📊 BASELINE 2: Global Median (Robust)")
print("=" * 60)

global_median = y_train.median()

baseline2_val_pred = np.full(len(y_val), global_median)
baseline2_test_pred = np.full(len(y_test), global_median)

baseline2_val_mae = mean_absolute_error(y_val, baseline2_val_pred)
baseline2_test_mae = mean_absolute_error(y_test, baseline2_test_pred)

baseline2_val_rmse = calculate_rmse(y_val, baseline2_val_pred)
baseline2_test_rmse = calculate_rmse(y_test, baseline2_test_pred)

print(f"   Global median: {global_median:,.0f}")
print(f"   Validation MAE:  {baseline2_val_mae:,.0f}")
print(f"   Validation RMSE: {baseline2_val_rmse:,.0f}")
print(f"   Test MAE:        {baseline2_test_mae:,.0f}")
print(f"   Test RMSE:       {baseline2_test_rmse:,.0f}")

# ============================================================================
# BASELINE 3: LAG-1 YEAR BASELINE (MOST IMPORTANT)
# ============================================================================

print("\n📊 BASELINE 3: Lag-1 Baseline (cutoff_lag1Y_L1Y)")
print("=" * 60)

if "cutoff_lag1Y_L1Y" not in X_train.columns:
    raise ValueError("cutoff_lag1Y_L1Y not found in features! Check Stage 2.")

baseline3_val_pred = X_val["cutoff_lag1Y_L1Y"].fillna(global_mean)
baseline3_test_pred = X_test["cutoff_lag1Y_L1Y"].fillna(global_mean)

baseline3_val_mae = mean_absolute_error(y_val, baseline3_val_pred)
baseline3_test_mae = mean_absolute_error(y_test, baseline3_test_pred)

baseline3_val_rmse = calculate_rmse(y_val, baseline3_val_pred)
baseline3_test_rmse = calculate_rmse(y_test, baseline3_test_pred)

print(f"   Validation MAE:  {baseline3_val_mae:,.0f}")
print(f"   Validation RMSE: {baseline3_val_rmse:,.0f}")
print(f"   Test MAE:        {baseline3_test_mae:,.0f}")
print(f"   Test RMSE:       {baseline3_test_rmse:,.0f}")

# ============================================================================
# BASELINE SUMMARY (SORTED BY VALIDATION MAE)
# ============================================================================

print("\n" + "=" * 80)
print("🎯 BASELINE SUMMARY (VALIDATION SET)")
print("=" * 80)

baselines = [
    ('Global Mean', baseline1_val_mae),
    ('Global Median', baseline2_val_mae),
    ('Lag-1 Baseline', baseline3_val_mae)
]

baselines_sorted = sorted(baselines, key=lambda x: x[1])

for i, (name, mae) in enumerate(baselines_sorted, 1):
    print(f"   {i}. {name:<20} MAE: {mae:,.0f}")

best_baseline_name, best_baseline_mae = baselines_sorted[0]

print(f"\n🏆 BEST BASELINE TO BEAT: {best_baseline_name} ({best_baseline_mae:,.0f} MAE)")
print(f"   Your ML model MUST beat this to be considered useful.")

print("\n✅ CELL 3 COMPLETE!")
print("=" * 80)



CELL 3: BASELINE MODELS (STRONGER & CORRECT BENCHMARKS)

📊 BASELINE 1: Global Mean (Simple Average)
   Global mean: 69,321
   Validation MAE:  41,955
   Validation RMSE: 53,105
   Test MAE:        60,419
   Test RMSE:       78,861

📊 BASELINE 2: Global Median (Robust)
   Global median: 61,038
   Validation MAE:  42,986
   Validation RMSE: 55,608
   Test MAE:        63,753
   Test RMSE:       83,397

📊 BASELINE 3: Lag-1 Baseline (cutoff_lag1Y_L1Y)
   Validation MAE:  27,366
   Validation RMSE: 41,221
   Test MAE:        44,505
   Test RMSE:       66,947

🎯 BASELINE SUMMARY (VALIDATION SET)
   1. Lag-1 Baseline       MAE: 27,366
   2. Global Mean          MAE: 41,955
   3. Global Median        MAE: 42,986

🏆 BEST BASELINE TO BEAT: Lag-1 Baseline (27,366 MAE)
   Your ML model MUST beat this to be considered useful.

✅ CELL 3 COMPLETE!


In [16]:
# ============================================================================
# STAGE 3 - CELL 4: TRAIN IMPROVED LIGHTGBM MODEL (TIME-SAFE, TUNED BASELINE)
# ============================================================================

import lightgbm as lgb
import time
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("\n" + "=" * 80)
print("CELL 4: TRAIN LIGHTGBM MODEL (IMPROVED & OPTIMIZED)")
print("=" * 80)

def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# ============================================================================
# 1. Initialize Improved LightGBM Model
# ============================================================================

print("\n🔧 INITIALIZING IMPROVED LIGHTGBM MODEL...")

lgbm_model = lgb.LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1.0,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
    objective='mae'
)

print(f"""
   ✔ n_estimators: 3000
   ✔ learning_rate: 0.03
   ✔ num_leaves: 63
   ✔ subsample: 0.85
   ✔ colsample_bytree: 0.85
   ✔ reg_alpha: 1.0
   ✔ reg_lambda: 2.0
   ✔ Early stopping: 200 rounds
   ✔ Objective: MAE (with Log-Transformed Target)
""")

# ============================================================================
# 2. Train Model with Early Stopping
# ============================================================================

print("\n⏳ TRAINING MODEL...")
start_time = time.time()

lgbm_model.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    eval_metric='l1',
    callbacks=[
        lgb.early_stopping(stopping_rounds=200, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

train_time = time.time() - start_time
print(f"\n✅ TRAINING COMPLETE in {train_time:.2f} seconds ({train_time/60:.2f} minutes).")

# ============================================================================
# 3. Generate Predictions
# ============================================================================

print("\n🔮 GENERATING PREDICTIONS (with expm1 inverse transform)...")

train_pred = np.expm1(lgbm_model.predict(X_train))
val_pred = np.expm1(lgbm_model.predict(X_val))
test_pred = np.expm1(lgbm_model.predict(X_test))

print("   ✔ Predictions ready!")

# ============================================================================
# 4. Evaluate Model Performance
# ============================================================================

print("\n" + "=" * 80)
print("📊 MODEL PERFORMANCE (MAE / RMSE / R²)")
print("=" * 80)

train_mae = mean_absolute_error(y_train, train_pred)
val_mae = mean_absolute_error(y_val, val_pred)
test_mae = mean_absolute_error(y_test, test_pred)

train_rmse = calculate_rmse(y_train, train_pred)
val_rmse = calculate_rmse(y_val, val_pred)
test_rmse = calculate_rmse(y_test, test_pred)

train_r2 = r2_score(y_train, train_pred)
val_r2 = r2_score(y_val, val_pred)
test_r2 = r2_score(y_test, test_pred)

print(f"""
🎯 MAE:
   • Train: {train_mae:,.0f}
   • Val:   {val_mae:,.0f}
   • Test:  {test_mae:,.0f}

📏 RMSE:
   • Train: {train_rmse:,.0f}
   • Val:   {val_rmse:,.0f}
   • Test:  {test_rmse:,.0f}

📈 R² SCORE:
   • Train: {train_r2:.4f}
   • Val:   {val_r2:.4f}
   • Test:  {test_r2:.4f}
""")

print("✅ CELL 4 COMPLETE!")
print("=" * 80)



CELL 4: TRAIN LIGHTGBM MODEL (IMPROVED & OPTIMIZED)

🔧 INITIALIZING IMPROVED LIGHTGBM MODEL...

   ✔ n_estimators: 3000
   ✔ learning_rate: 0.03
   ✔ num_leaves: 63
   ✔ subsample: 0.85
   ✔ colsample_bytree: 0.85
   ✔ reg_alpha: 1.0
   ✔ reg_lambda: 2.0
   ✔ Early stopping: 200 rounds
   ✔ Objective: MAE (with Log-Transformed Target)


⏳ TRAINING MODEL...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033364 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3105
[LightGBM] [Info] Number of data points in the train set: 137755, number of used features: 26
[LightGBM] [Info] Start training from score 11.019268
Training until validation scores don't improve for 200 rounds
[50]	valid_0's l1: 0.346549
[100]	valid_0's l1: 0.289435
[150]	valid_0's l1: 0.276258
[200]	valid_0's l1: 0.2728
[250]	valid_0's l1: 0.272429
[300]	valid_0's l1: 0.271505
[350]	valid_0's l1: 0.268226
[400]	valid_0's l1: 0.266118
[450]

In [23]:
# ============================================================================
# STAGE 3 - CELL 5 (FINAL SAFE VERSION) 
# - Map back identifiers using target-encoding reversal (Nearest Neighbor)
# - Advanced diagnostics: residuals, RMSE gap, slice MAE, drift, calibration
# - Memory-safe: NO cartesian joins
# ============================================================================

import os
import pandas as pd
import numpy as np
import warnings
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import NearestNeighbors
from scipy.stats import ks_2samp
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
REPORT_DIR = "diagnostics_reports"
os.makedirs(REPORT_DIR, exist_ok=True)

# -------------------------
# Paths - adjust if required
# -------------------------
RAW_PATH   = "kcet_ml_project/data/df_optimized.csv"
TRAIN_PROC = "kcet_ml_project/data/stage2_v2_corrected/train_stage2_final.csv"
VAL_PROC   = "kcet_ml_project/data/stage2_v2_corrected/val_stage2_final.csv"
TEST_PROC  = "kcet_ml_project/data/stage2_v2_corrected/test_stage2_final.csv"

# -------------------------
# Utility functions
# -------------------------
def safe_read_csv(p):
    if not os.path.exists(p):
        raise FileNotFoundError(f"File not found: {p}\nIf files expired in this environment, re-upload them and re-run this cell.")
    return pd.read_csv(p)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# -------------------------
# 1) Load data
# -------------------------
print("Loading files...")
df_raw   = safe_read_csv(RAW_PATH)     # raw before stage2 (has College_Code, Branch, etc.)
train_df = safe_read_csv(TRAIN_PROC)   # processed stage2 train
val_df   = safe_read_csv(VAL_PROC)
test_df  = safe_read_csv(TEST_PROC)
print("Loaded shapes:", df_raw.shape, train_df.shape, val_df.shape, test_df.shape)

# -------------------------
# 2) Quick checks
# -------------------------
required_proc_cols = ['College_Code_target_enc', 'College_Branch_target_enc']
for c in required_proc_cols:
    if c not in train_df.columns:
        raise KeyError(f"Processed file missing required column: {c}. Stage2 must have produced target-encodings.")

# -------------------------
# 3) Build raw-side encoding features to match processed encodings
#    - college_mean_cutoff (per College_Code)
#    - college_branch_mean_cutoff (per College_Code + Branch)
# -------------------------
print("Computing raw aggregated encodings (college & college+branch means)...")
# Use Cutoff_Rank as value that Stage2 likely used to compute target-encodings (average cutoff)
raw_college_mean = df_raw.groupby('College_Code', as_index=False)['Cutoff_Rank'].mean().rename(columns={'Cutoff_Rank':'raw_college_mean_cutoff'})
raw_college_branch_mean = df_raw.groupby(['College_Code','Branch'], as_index=False)['Cutoff_Rank'].mean().rename(columns={'Cutoff_Rank':'raw_college_branch_mean_cutoff'})

# Merge the two into a RHS encoding table
raw_enc = raw_college_branch_mean.merge(raw_college_mean, on='College_Code', how='left')
# Keep only necessary columns (unique combos)
raw_enc = raw_enc.drop_duplicates(subset=['College_Code','Branch']).reset_index(drop=True)
print("Raw encodings shape:", raw_enc.shape)

# -------------------------
# 4) Prepare processed encoding vectors (train/val/test)
# -------------------------
def collect_proc_enc(proc_df):
    # use float encodings present in processed df
    return proc_df[['College_Code_target_enc','College_Branch_target_enc']].copy()

X_proc_train = collect_proc_enc(train_df)
X_proc_val   = collect_proc_enc(val_df)
X_proc_test  = collect_proc_enc(test_df)

# -------------------------
# 5) Fit NearestNeighbors on raw_enc vectors (2D)
#    We'll match processed (college_enc, college_branch_enc) -> (raw_college_mean_cutoff, raw_college_branch_mean_cutoff)
# -------------------------
# Prepare raw matrix (2 columns)
raw_matrix = raw_enc[['raw_college_mean_cutoff','raw_college_branch_mean_cutoff']].fillna(-1).values.astype(float)

# Use small leaf_size; n_neighbors=1 for exact nearest
nn = NearestNeighbors(n_neighbors=1, metric='euclidean', n_jobs=-1)
nn.fit(raw_matrix)

# Helper to match a proc-encoding vector to college_code & branch
def map_proc_to_raw(proc_enc_df, proc_name="proc"):
    # Build query vectors from processed target encodings
    # Note: stage2 produced target-enc floats which approximate raw means;
    # We will map them using NN. If one of the two encodings is NaN, fallback to college-only match.
    q = proc_enc_df[['College_Code_target_enc','College_Branch_target_enc']].fillna(-1).values.astype(float)
    # If q contains large number scales that differ from raw_matrix scale, normalize both sides to z-score per-column:
    # Compute per-column z-scores to make NN robust to scale differences.
    # Stack raw and q to compute same scaling
    stacked = np.vstack([raw_matrix, q])
    col_mean = stacked.mean(axis=0)
    col_std = stacked.std(axis=0) + 1e-9
    raw_scaled = (raw_matrix - col_mean) / col_std
    q_scaled = (q - col_mean) / col_std

    neigh = NearestNeighbors(n_neighbors=1, metric='euclidean', n_jobs=-1).fit(raw_scaled)
    dists, idxs = neigh.kneighbors(q_scaled, return_distance=True)
    idxs = idxs.flatten()
    dists = dists.flatten()

    # Build mapping results
    mapped = raw_enc.iloc[idxs].reset_index(drop=True).copy()
    mapped['nn_distance'] = dists
    # Attach original proc index
    mapped.index = proc_enc_df.index
    return mapped

print("Mapping processed encodings -> raw identifiers using nearest-neighbor lookup (fast, memory-safe)...")
mapped_train = map_proc_to_raw(X_proc_train, "train")
mapped_val   = map_proc_to_raw(X_proc_val, "val")
mapped_test  = map_proc_to_raw(X_proc_test, "test")

# -------------------------
# 6) Attach mapped identifiers to processed frames (for evaluation only)
# -------------------------
def attach_identifiers(proc_df, mapped_df):
    out = proc_df.reset_index(drop=True).copy()
    # mapped_df contains College_Code, Branch, and raw means
    # add columns safely (avoid collisions)
    for col in ['College_Code','Branch','raw_college_mean_cutoff','raw_college_branch_mean_cutoff','nn_distance']:
        out[col] = mapped_df[col].values
    return out

merged_train = attach_identifiers(train_df, mapped_train)
merged_val   = attach_identifiers(val_df, mapped_val)
merged_test  = attach_identifiers(test_df, mapped_test)

print("Attached mapped identifiers. Example nn distances (train):")
print(mapped_train['nn_distance'].describe())

# sanity: report proportion of high-distance matches (flag if mapping unreliable)
threshold = np.percentile(mapped_train['nn_distance'].values, 95)
n_high = (mapped_train['nn_distance'] > threshold).sum()
print(f"Warning threshold (95th pct) = {threshold:.4f}. Matches above threshold in train: {n_high}/{len(mapped_train)}")
# Save mapping diagnostics
mapped_train[['nn_distance']].describe().to_csv(os.path.join(REPORT_DIR,'mapping_nn_stats_train.csv'))

# -------------------------
# 7) Diagnostics: compute predictions/residuals if model exists
# -------------------------
# The cell expects a trained model in memory named 'final_model' or 'lgbm_model' from Stage 3 training.
model = globals().get('final_model', globals().get('lgbm_model', None))
if model is None:
    print("No trained model found in memory (final_model / lgbm_model). Skipping prediction-based diagnostics.")
else:
    print("Model found. Computing predictions and evaluation metrics.")
    # Prepare X matrices for model inference: use the same feature set used for training.
    # We assume processed train_df/val_df/test_df contain exactly the features used for training (except Cutoff_Rank).
    model_features = [c for c in train_df.columns if c != 'Cutoff_Rank']
    # Defensive: ensure model_features exist in merged_* (they should)
    X_train = merged_train[model_features]
    X_val   = merged_val[model_features]
    X_test  = merged_test[model_features]

    # detect if training used log1p target (heuristic)
    use_log = 'y_train_log' in globals() or False

    def model_predict(X):
        pred = model.predict(X)
        return np.expm1(pred) if use_log else pred

    y_train = train_df['Cutoff_Rank'].values
    y_val   = val_df['Cutoff_Rank'].values
    y_test  = test_df['Cutoff_Rank'].values

    yhat_train = model_predict(X_train)
    yhat_val   = model_predict(X_val)
    yhat_test  = model_predict(X_test)

    # Global metrics
    print("\nGLOBAL METRICS")
    print("Train MAE:", mean_absolute_error(y_train, yhat_train))
    print("Val   MAE:", mean_absolute_error(y_val, yhat_val))
    print("Test  MAE:", mean_absolute_error(y_test, yhat_test))
    print("Train RMSE:", rmse(y_train, yhat_train))
    print("Val RMSE:", rmse(y_val, yhat_val))
    print("Test RMSE:", rmse(y_test, yhat_test))
    print("Train R2:", r2_score(y_train, yhat_train))
    print("Val R2:", r2_score(y_val, yhat_val))
    print("Test R2:", r2_score(y_test, yhat_test))

    # Residual distributions & RMSE gap
    res_val = y_val - yhat_val
    res_test = y_test - yhat_test
    val_rmse = rmse(y_val, yhat_val)
    test_rmse = rmse(y_test, yhat_test)
    gap_ratio = test_rmse / (val_rmse + 1e-9)
    print(f"RMSE gap ratio Test/Val = {gap_ratio:.2f}x (Val={val_rmse:.1f}, Test={test_rmse:.1f})")
    if gap_ratio > 3:
        print("🚨 RMSE gap > 3: investigate covariate shift / leakage / overfitting.")
    else:
        print("✅ RMSE gap acceptable.")

    # Save global metrics
    metrics = {
        'train_mae': mean_absolute_error(y_train, yhat_train),
        'val_mae': mean_absolute_error(y_val, yhat_val),
        'test_mae': mean_absolute_error(y_test, yhat_test),
        'train_rmse': rmse(y_train, yhat_train),
        'val_rmse': rmse(y_val, yhat_val),
        'test_rmse': rmse(y_test, yhat_test),
        'rmse_gap': gap_ratio
    }
    pd.Series(metrics).to_csv(os.path.join(REPORT_DIR, "global_metrics.csv"))

    # -------------------------
    # Feature importance + permutation importance (val)
    # -------------------------
    try:
        fi = pd.DataFrame({
            'feature': X_train.columns,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        fi.to_csv(os.path.join(REPORT_DIR,'feature_importance_builtin.csv'), index=False)
        print("Saved built-in feature importance.")
    except Exception as e:
        print("Built-in FI error:", e)

    try:
        perm = permutation_importance(model, X_val, y_val, n_repeats=8, random_state=42, n_jobs=-1)
        perm_df = pd.DataFrame({
            'feature': X_val.columns,
            'perm_mean': perm.importances_mean,
            'perm_std': perm.importances_std
        }).sort_values('perm_mean', ascending=False)
        perm_df.to_csv(os.path.join(REPORT_DIR,'permutation_importance_val.csv'), index=False)
        print("Saved permutation importance (val).")
    except Exception as e:
        print("Permutation importance skipped/failed:", e)

    # -------------------------
    # 8) Slice-wise MAE on TEST set (advanced)
    # -------------------------
    def compute_slice_mae(df_merged, y_true_arr, y_pred_arr, col, top_n=20):
        if col not in df_merged.columns:
            print(f" - Skip slice {col}: not present")
            return None
        tmp = pd.DataFrame({
            col: df_merged[col].astype(str),
            'y_true': y_true_arr,
            'y_pred': y_pred_arr
        }).dropna(subset=[col])
        if tmp.empty:
            print(" - No rows for", col)
            return None
        grouped = tmp.groupby(col).apply(lambda g: mean_absolute_error(g['y_true'], g['y_pred']))
        grouped = grouped.sort_values(ascending=False)
        out = grouped.reset_index().rename(columns={0:'mae', col:'id'})  # older pandas mapping
        out.to_csv(os.path.join(REPORT_DIR, f'slice_mae_test_{col}.csv'), index=False)
        print(f"Saved slice MAE for {col} ({len(grouped)} groups). Top {top_n} saved.")
        return out

    slice_cols = ['College_Code','College_Name','Branch','Category','Category_Simplified','Exam_Type','Year','Round']
    slice_results = {}
    for c in slice_cols:
        slice_results[c] = compute_slice_mae(merged_test, y_test, yhat_test, c, top_n=50)

    # -------------------------
    # 9) Drift test (KS) between Val and Test for top N features
    # -------------------------
    try:
        X_val_small = X_val.sample(n=min(50000, len(X_val)), random_state=42) if len(X_val)>50000 else X_val
        X_test_small = X_test.sample(n=min(50000, len(X_test)), random_state=42) if len(X_test)>50000 else X_test
        ks_results = []
        for f in X_val_small.columns:
            v = X_val_small[f].dropna()
            t = X_test_small[f].dropna()
            if len(v)>0 and len(t)>0:
                stat, p = ks_2samp(v, t)
                ks_results.append((f, float(stat), float(p)))
        ks_df = pd.DataFrame(ks_results, columns=['feature','ks_stat','p']).sort_values('ks_stat', ascending=False)
        ks_df.to_csv(os.path.join(REPORT_DIR,'ks_val_vs_test.csv'), index=False)
        print("Saved KS drift report (val vs test).")
    except Exception as e:
        print("KS test failed/skipped:", e)

    # -------------------------
    # 10) Calibration (test quantiles)
    # -------------------------
    try:
        buckets = pd.qcut(y_test, 10, labels=False, duplicates='drop')
        calib = pd.DataFrame({'true': y_test, 'pred': yhat_test, 'bucket': buckets})
        calib_summary = calib.groupby('bucket').agg(true_median=('true','median'), pred_median=('pred','median'), count=('true','count'))
        calib_summary.to_csv(os.path.join(REPORT_DIR,'calibration_test_by_quantile.csv'))
        print("Saved calibration summary.")
    except Exception as e:
        print("Calibration step failed/skipped:", e)

print("\nCELL 5 (final) complete. Reports saved to:", REPORT_DIR)


Loading files...
Loaded shapes: (270062, 23) (137755, 33) (60681, 33) (71626, 33)
Computing raw aggregated encodings (college & college+branch means)...
Raw encodings shape: (4776, 4)
Mapping processed encodings -> raw identifiers using nearest-neighbor lookup (fast, memory-safe)...
Attached mapped identifiers. Example nn distances (train):
count    137755.000000
mean          0.037880
std           0.026915
min           0.000162
25%           0.019505
50%           0.032511
75%           0.049253
max           0.385772
Name: nn_distance, dtype: float64
Warning threshold (95th pct) = 0.0856. Matches above threshold in train: 6885/137755
Model found. Computing predictions and evaluation metrics.

GLOBAL METRICS
Train MAE: 11482.757124886868
Val   MAE: 18860.418500215615
Test  MAE: 31949.222560217248
Train RMSE: 18405.195778483085
Val RMSE: 28035.020152154444
Test RMSE: 45473.87259276978
Train R2: 0.8340973173263418
Val R2: 0.7055482483519284
Test R2: 0.5500890990462258
RMSE gap ratio T

In [32]:
# ============================================================================
# STAGE 3 - CELL 6 (FINAL): OPTUNA TUNING + FINAL MODEL TRAINING
# ============================================================================

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib, os

print("\n" + "="*80)
print("CELL 6: OPTUNA TUNING + FINAL MODEL TRAINING (COMPATIBLE VERSION)")
print("="*80)

MODEL_DIR = "models"
REPORT_DIR = "model_reports"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ---------------- DATA ----------------
X_train = train_df.drop(columns=["Cutoff_Rank"])
y_train = train_df["Cutoff_Rank"]

X_val = val_df.drop(columns=["Cutoff_Rank"])
y_val = val_df["Cutoff_Rank"]

X_test = test_df.drop(columns=["Cutoff_Rank"])
y_test = test_df["Cutoff_Rank"]

# ---------------- OPTUNA OBJECTIVE ----------------
def objective(trial):
    params = {
        "objective": "regression",
        "metric": "mae",
        "boosting_type": "gbdt",
        "random_state": 42,
        "n_jobs": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.12),
        "num_leaves": trial.suggest_int("num_leaves", 24, 128),
        "max_depth": trial.suggest_int("max_depth", 5, 18),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
        "n_estimators": trial.suggest_int("n_estimators", 300, 1800),
    }

    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="mae",
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=0)
        ]
    )

    preds = model.predict(X_val)
    return mean_absolute_error(y_val, preds)

# ---------------- RUN OPTUNA ----------------
print("\n🔍 Running Optuna tuning (50 trials)...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n🏆 Best Params:")
print(study.best_params)
print("Best Validation MAE:", study.best_value)

best_params = study.best_params

# ---------------- FINAL MODEL TRAINING ----------------
print("\n⚡ Training FINAL model on TRAIN + VAL...")

final_model = lgb.LGBMRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    pd.concat([X_train, X_val]),
    pd.concat([y_train, y_val]),
    callbacks=[
        lgb.log_evaluation(period=50)
    ]
)

# ---------------- PREDICTIONS ----------------
train_pred = final_model.predict(X_train)
val_pred = final_model.predict(X_val)
test_pred = final_model.predict(X_test)

# ---------------- METRICS ----------------
metrics = {
    "train_mae": mean_absolute_error(y_train, train_pred),
    "val_mae": mean_absolute_error(y_val, val_pred),
    "test_mae": mean_absolute_error(y_test, test_pred),
    "train_rmse": np.sqrt(mean_squared_error(y_train, train_pred)),
    "val_rmse": np.sqrt(mean_squared_error(y_val, val_pred)),
    "test_rmse": np.sqrt(mean_squared_error(y_test, test_pred)),
    "train_r2": r2_score(y_train, train_pred),
    "val_r2": r2_score(y_val, val_pred),
    "test_r2": r2_score(y_test, test_pred),
}

print("\n📊 FINAL PERFORMANCE:")
for k,v in metrics.items():
    print(f"{k}: {v:,.4f}")

pd.Series(metrics).to_csv(os.path.join(REPORT_DIR, "optuna_final_metrics.csv"))

# ---------------- SAVE MODEL ----------------
joblib.dump(final_model, os.path.join(MODEL_DIR, "lgbm_optuna.pkl"))
print("\n✔ Saved final model → models/lgbm_optuna.pkl")
print("✔ Saved metrics → model_reports/optuna_final_metrics.csv")

print("\n✅ CELL 6 COMPLETE")
print("="*80)


[I 2025-11-14 02:38:51,394] A new study created in memory with name: no-name-7ca6555c-e84f-43d4-a51b-f7fd739e8370



CELL 6: OPTUNA TUNING + FINAL MODEL TRAINING (COMPATIBLE VERSION)

🔍 Running Optuna tuning (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005839 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3105
[LightGBM] [Info] Number of data points in the train set: 137755, number of used features: 26
[LightGBM] [Info] Start training from score 69320.939842
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

Best trial: 0. Best value: 19753.6:   2%|▏         | 1/50 [00:11<09:16, 11.36s/it]

...

[I 2025-11-14 02:43:53,182] Trial 47 finished with value: 19283.779394185258 and parameters: {'learning_rate': 0.06823851571372408, 'num_leaves': 71, 'max_depth': 16, 'min_child_samples': 57, 'subsample': 0.6990284152138707, 'colsample_bytree': 0.6573580485476933, 'reg_alpha': 1.3331180260630378, 'reg_lambda': 0.9567318977561645, 'n_estimators': 602}. Best is trial 21 with value: 19067.917083769924.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019380 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3105
[LightGBM] [Info] Number of data points in the train set: 137755, number of used features: 26
[LightGBM] [Info] Start training from score 69320.939842
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

Best trial: 21. Best value: 19067.9:  98%|█████████▊| 49/50 [05:09<00:07,  7.19s/it]

[I 2025-11-14 02:44:01,083] Trial 48 finished with value: 19303.90301824475 and parameters: {'learning_rate': 0.07011745139604678, 'num_leaves': 71, 'max_depth': 9, 'min_child_samples': 56, 'subsample': 0.6960246270803561, 'colsample_bytree': 0.7067175103149549, 'reg_alpha': 0.8378990751560464, 'reg_lambda': 1.2977910282804128, 'n_estimators': 591}. Best is trial 21 with value: 19067.917083769924.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015535 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3105
[LightGBM] [Info] Number of data points in the train set: 137755, number of used features: 26
[LightGBM] [Info] Start training from score 69320.939842
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

Best trial: 21. Best value: 19067.9: 100%|██████████| 50/50 [05:16<00:00,  6.34s/it]


[I 2025-11-14 02:44:08,341] Trial 49 finished with value: 19553.86797908554 and parameters: {'learning_rate': 0.0712631996727503, 'num_leaves': 54, 'max_depth': 9, 'min_child_samples': 57, 'subsample': 0.7583272234196523, 'colsample_bytree': 0.7095577585096469, 'reg_alpha': 1.3394070387919006, 'reg_lambda': 1.3620088548114375, 'n_estimators': 620}. Best is trial 21 with value: 19067.917083769924.

🏆 Best Params:
{'learning_rate': 0.06813001976292711, 'num_leaves': 83, 'max_depth': 12, 'min_child_samples': 24, 'subsample': 0.6893244572318318, 'colsample_bytree': 0.6076448024677207, 'reg_alpha': 1.1672594470391144, 'reg_lambda': 0.3046199845461245, 'n_estimators': 773}
Best Validation MAE: 19067.917083769924

⚡ Training FINAL model on TRAIN + VAL...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012714 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info

In [36]:
# =========================
# CELL 7: MULTI-MODEL TRAIN & EVAL (LightGBM / XGBoost / CatBoost)
# Backwards-compatible training for older libraries
# =========================

import os
import time
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Config - paths (adjust if needed)
BASE = 'kcet_ml_project/data/stage2_v2_corrected'
X_TRAIN_F = os.path.join(BASE, 'X_train_stage2.csv')
X_VAL_F   = os.path.join(BASE, 'X_val_stage2.csv')
X_TEST_F  = os.path.join(BASE, 'X_test_stage2.csv')
TRAIN_F   = os.path.join(BASE, 'train_stage2_final.csv')   # contains Cutoff_Rank (target) if needed
VAL_F     = os.path.join(BASE, 'val_stage2_final.csv')
TEST_F    = os.path.join(BASE, 'test_stage2_final.csv')

OUT_DIR = 'model_reports'
os.makedirs(OUT_DIR, exist_ok=True)

# Load feature matrices and targets (assumes same ordering)
print("Loading X_train/X_val/X_test ...")
X_train = pd.read_csv(X_TRAIN_F)
X_val   = pd.read_csv(X_VAL_F)
X_test  = pd.read_csv(X_TEST_F)

# If Cutoff_Rank target is stored in train/val/test files:
def load_targets_if_present():
    y_train = y_val = y_test = None
    if os.path.exists(TRAIN_F):
        train_df = pd.read_csv(TRAIN_F)
        if 'Cutoff_Rank' in train_df.columns:
            y_train = train_df['Cutoff_Rank'].values
    if os.path.exists(VAL_F):
        val_df = pd.read_csv(VAL_F)
        if 'Cutoff_Rank' in val_df.columns:
            y_val = val_df['Cutoff_Rank'].values
    if os.path.exists(TEST_F):
        test_df = pd.read_csv(TEST_F)
        if 'Cutoff_Rank' in test_df.columns:
            y_test = test_df['Cutoff_Rank'].values
    return y_train, y_val, y_test

y_train, y_val, y_test = load_targets_if_present()
if y_train is None:
    raise RuntimeError("Target `Cutoff_Rank` not found in train file. Ensure targets are available.")

print(f"Shapes: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")
print(f"Targets: y_train={y_train.shape}, y_val={(y_val.shape if y_val is not None else None)}, y_test={(y_test.shape if y_test is not None else None)}")

# Convert Exam_Type mapping if raw df exists (reconstruct)
raw_mapping = None
raw_path = 'kcet_ml_project/data/df_optimized.csv'
if os.path.exists(raw_path):
    df_raw = pd.read_csv(raw_path)
    if 'Exam_Type' in df_raw.columns:
        raw_unique = sorted(df_raw['Exam_Type'].unique())
        # Assumption: processed used alphabetical LabelEncoder mapping
        raw_map = {v: i for i, v in enumerate(sorted(raw_unique))}
        raw_mapping = raw_map
        print("Reconstructed Exam_Type mapping:", raw_mapping)

# Feature names
FEATURES = X_train.columns.tolist()

# --------------------------
# Helper: evaluate
# --------------------------
def evaluate_preds(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    return {'mae': float(mae), 'rmse': float(rmse), 'r2': float(r2)}

# --------------------------
# 1) LightGBM training (safe for older versions)
# --------------------------
try:
    import lightgbm as lgb
    print("LightGBM version:", lgb.__version__)
    lgb_results = {}
    lgb_model = None

    lgb_params = {
        'objective': 'regression',
        'metric': 'l1',
        'verbosity': -1,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'feature_fraction': 0.85,
        'bagging_fraction': 0.85,
        'bagging_freq': 1,
        'lambda_l1': 1.0,
        'lambda_l2': 2.0,
        'seed': 42
    }

    # Build datasets
    dtrain = lgb.Dataset(X_train[FEATURES], label=y_train)
    dval = lgb.Dataset(X_val[FEATURES], label=y_val, reference=dtrain)

    # Use lgb.train which is version-stable and supports early_stopping_rounds
    print("Training LightGBM via lgb.train (safe API)...")
    t0 = time.time()
    lgb_model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dtrain, dval],
        valid_names=['train','valid'],
        early_stopping_rounds=200,
        verbose_eval=100
    )
    t1 = time.time()
    print(f"LightGBM trained in {(t1-t0):.1f}s, best_iter={lgb_model.best_iteration}")

    # Predict and evaluate
    p_train = lgb_model.predict(X_train[FEATURES], num_iteration=lgb_model.best_iteration)
    p_val   = lgb_model.predict(X_val[FEATURES], num_iteration=lgb_model.best_iteration)
    p_test  = lgb_model.predict(X_test[FEATURES], num_iteration=lgb_model.best_iteration) if y_test is not None else None

    lgb_results['train'] = evaluate_preds(y_train, p_train)
    lgb_results['val']   = evaluate_preds(y_val, p_val)
    if p_test is not None:
        lgb_results['test']  = evaluate_preds(y_test, p_test)

    # Save model & feature importance
    joblib.dump(lgb_model, os.path.join(OUT_DIR, 'lgb_model.pkl'))
    fi = pd.DataFrame({'feature': FEATURES, 'importance': lgb_model.feature_importance(importance_type='gain')})
    fi.sort_values('importance', ascending=False).to_csv(os.path.join(OUT_DIR, 'lgb_feature_importance.csv'), index=False)

    print("LightGBM results:", lgb_results)
except Exception as e:
    print("ERROR training LightGBM:", e)
    lgb_results = None

# --------------------------
# 2) XGBoost training (use xgboost.train via DMatrix for compatibility)
# --------------------------
try:
    import xgboost as xgb
    print("XGBoost version:", xgb.__version__)
    xgb_results = {}
    xgb_model = None

    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'eta': 0.05,
        'max_depth': 8,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'lambda': 2.0,
        'alpha': 1.0,
        'seed': 42,
        'verbosity': 1
    }

    dtrain_x = xgb.DMatrix(X_train[FEATURES], label=y_train, feature_names=FEATURES)
    dval_x   = xgb.DMatrix(X_val[FEATURES], label=y_val, feature_names=FEATURES)
    watchlist = [(dtrain_x, 'train'), (dval_x, 'valid')]

    print("Training XGBoost via xgb.train (safe API)...")
    t0 = time.time()
    xgb_model = xgb.train(
        xgb_params,
        dtrain_x,
        num_boost_round=2000,
        evals=watchlist,
        early_stopping_rounds=200,
        verbose_eval=100
    )
    t1 = time.time()
    print(f"XGBoost trained in {(t1-t0):.1f}s, best_ntree_limit={xgb_model.best_ntree_limit}")

    p_train = xgb_model.predict(dtrain_x, ntree_limit=xgb_model.best_ntree_limit)
    p_val   = xgb_model.predict(dval_x,   ntree_limit=xgb_model.best_ntree_limit)
    p_test  = xgb_model.predict(xgb.DMatrix(X_test[FEATURES]), ntree_limit=xgb_model.best_ntree_limit) if y_test is not None else None

    xgb_results['train'] = evaluate_preds(y_train, p_train)
    xgb_results['val']   = evaluate_preds(y_val, p_val)
    if p_test is not None:
        xgb_results['test'] = evaluate_preds(y_test, p_test)

    joblib.dump(xgb_model, os.path.join(OUT_DIR, 'xgb_model.pkl'))
    # feature importance
    fmap = xgb_model.get_score(importance_type='gain')
    fi_x = pd.DataFrame([{'feature': k, 'importance': v} for k, v in fmap.items()]).sort_values('importance', ascending=False)
    fi_x.to_csv(os.path.join(OUT_DIR, 'xgb_feature_importance.csv'), index=False)

    print("XGBoost results:", xgb_results)
except Exception as e:
    print("ERROR training XGBoost:", e)
    xgb_results = None

# --------------------------
# 3) CatBoost training (safe fit API)
# --------------------------
try:
    from catboost import CatBoostRegressor, Pool
    print("CatBoost version: (catboost import succeeded)")
    cat_results = {}
    cat_model = None

    cat_params = {
        'iterations': 2000,
        'learning_rate': 0.03,
        'depth': 8,
        'l2_leaf_reg': 3,
        'loss_function': 'MAE',
        'random_seed': 42,
        'verbose': 100
    }

    # CatBoost can auto-detect categorical features by name, but here we assume all numeric
    train_pool = Pool(X_train[FEATURES], label=y_train)
    val_pool = Pool(X_val[FEATURES], label=y_val)

    cat_model = CatBoostRegressor(**cat_params)
    print("Training CatBoost via CatBoostRegressor.fit ...")
    t0 = time.time()
    # fit supports eval_set in all modern versions - if older, we wrap in try/except
    try:
        cat_model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=200, use_best_model=True)
    except TypeError:
        # fallback if early_stopping_rounds not accepted (very old versions)
        cat_model.fit(train_pool, eval_set=val_pool, verbose=100)
    t1 = time.time()
    print(f"CatBoost trained in {(t1-t0):.1f}s, best_iteration={cat_model.get_best_iteration()}")

    p_train = cat_model.predict(X_train[FEATURES])
    p_val   = cat_model.predict(X_val[FEATURES])
    p_test  = cat_model.predict(X_test[FEATURES]) if y_test is not None else None

    cat_results['train'] = evaluate_preds(y_train, p_train)
    cat_results['val']   = evaluate_preds(y_val, p_val)
    if p_test is not None:
        cat_results['test']  = evaluate_preds(y_test, p_test)

    joblib.dump(cat_model, os.path.join(OUT_DIR, 'cat_model.pkl'))
    fi_cat = pd.DataFrame({'feature': FEATURES, 'importance': cat_model.get_feature_importance()})
    fi_cat.sort_values('importance', ascending=False).to_csv(os.path.join(OUT_DIR, 'cat_feature_importance.csv'), index=False)

    print("CatBoost results:", cat_results)
except Exception as e:
    print("ERROR training CatBoost:", e)
    cat_results = None

# --------------------------
# Save summary metrics
# --------------------------
summary = {
    'lightgbm': lgb_results,
    'xgboost': xgb_results,
    'catboost': cat_results,
    'features': FEATURES
}
with open(os.path.join(OUT_DIR, 'multi_model_metrics.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=float)

print("\nAll done. Metrics written to:", os.path.join(OUT_DIR, 'multi_model_metrics.json'))


Loading X_train/X_val/X_test ...
Shapes: X_train=(137755, 32), X_val=(60681, 32), X_test=(71626, 32)
Targets: y_train=(137755,), y_val=(60681,), y_test=(71626,)
Reconstructed Exam_Type mapping: {'CET': 0, 'COMEDK': 1}
LightGBM version: 4.6.0
Training LightGBM via lgb.train (safe API)...
ERROR training LightGBM: train() got an unexpected keyword argument 'early_stopping_rounds'
XGBoost version: 3.1.1
Training XGBoost via xgb.train (safe API)...
[0]	train-mae:35944.24465	valid-mae:40613.39140
[100]	train-mae:12664.85275	valid-mae:19218.49870
[200]	train-mae:11784.01140	valid-mae:18912.24056
[300]	train-mae:11260.51859	valid-mae:18845.31704
[400]	train-mae:10840.18722	valid-mae:18842.33140
[500]	train-mae:10500.97396	valid-mae:18850.96040
[600]	train-mae:10191.17575	valid-mae:18865.52680
[637]	train-mae:10088.17636	valid-mae:18872.87591
ERROR training XGBoost: 'Booster' object has no attribute 'best_ntree_limit'
CatBoost version: (catboost import succeeded)
Training CatBoost via CatBoostR

In [ ]:
import lightgbm as lgb
import xgboost as xgb

print(lgb.__version__, xgb.__version__, cb.__version__)


4.6.0 3.1.1 1.2.8
